In [ ]:
# These are default values. Papermill will overwrite these at runtime.
question = "What are the prerequisites for CPT S 223?"
student_context = {"major": "Computer Science", "completed_courses": [], "credits_completed": 0}
use_rag = True


In [ ]:
import sys, os
from pathlib import Path
from dotenv import load_dotenv

base_dir = Path("/Volumes/asus/VC-n8n-440")
sys.path.insert(0, str(base_dir / "prompt-search" / "src"))
load_dotenv(base_dir / "prompt-search" / ".env")

index_dir = str(base_dir / "prompt-search" / "data" / "domain")


In [ ]:
import json, os, time
from retrieval.retriever import CourseRetriever
from retrieval.context_builder import ContextBuilder
from llm.claude_client import ClaudeClient
from llm.cache import ResponseCache

start_time = time.time()

# Papermill passes dicts as JSON strings
if isinstance(student_context, str):
    student_context = json.loads(student_context)

retriever = CourseRetriever(index_dir=index_dir)
builder   = ContextBuilder(retriever, top_k=5)
cache     = ResponseCache(db_path=str(base_dir / "prompt-search" / "data" / "cache" / "responses.db"))
client    = ClaudeClient(model="claude-haiku-4-5", api_key=os.getenv("ANTHROPIC_API_KEY"))

# Build student context block
student_block = ""
completed = student_context.get("completed_courses", [])
credits_done = student_context.get("credits_completed", 0)
major = student_context.get("major", "")
if completed:
    student_block += f"Student completed courses: {', '.join(completed)}\n"
if credits_done:
    student_block += f"Credits completed: {credits_done}\n"
if major and major != "Undeclared":
    student_block += f"Student major: {major}\n"

# Build prompt via RAG
base_prompt = student_block if student_block else ""
prompt, sources = builder.build(question, base_prompt=base_prompt)

# Call Claude (with cache)
cached = cache.get(prompt, model=client.model, temperature=0.0)
if cached:
    answer = cached
else:
    answer = client.generate(prompt, temperature=0.0, max_tokens=400)
    cache.set(prompt, model=client.model, temperature=0.0, response=answer)

source_codes = [s["course_code"] for s in sources]

output = {
    "answer": answer,
    "sources": source_codes,
    "metadata": {
        "used_rag": use_rag,
        "latency_seconds": round(time.time() - start_time, 2)
    }
}

import tempfile
temp_path = os.path.join(tempfile.gettempdir(), "llm_response.json")
with open(temp_path, "w") as f:
    json.dump(output, f)
print(json.dumps(output))
